**Notebook purposes:** build a standardized preprocessing function to execute baseline cleaning tasks across all datasets. 

In [1]:
import pandas as pd
import yaml
import openpyxl

In [2]:
# This code make possible avoid hardcoding the content (the path in this case) of the config.yaml file in the code, so that it can be easily modified without changing the code itself.
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [3]:
P_A1_25 = pd.read_excel(cfg["P_A1_25_path"], engine="openpyxl")

---

### Sanity Check.

After overview all raw files. They don't have same number or names columns.

In [4]:
P_A1_25.shape

(23444, 14)

In [5]:
# Checking whether the first and last DataFrame rows match the source database records.
P_A1_25.head(5) 

,\n \n Núm Albarán,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Agua\n 2\n AGUA RECICLA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3,Aditivo\n 4\n AURAMIX 269
0,36031,22/01/2026,SECOMD-200/S/4,6.0,1198.0,0.0,11315.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Dosif. m3,NaN,NaN,NaN,199.7,0.0,1885.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,36030,22/01/2026,HM-20/B/20/I/1,6.0,1213.0,0.0,7337.0,2116.0,3247.0,552.0,0.0,0.0,0.0,12.0
4,Dosif. m3,NaN,NaN,NaN,202.2,0.0,1222.8,352.7,541.2,92.0,0.0,0.0,0.0,2.0


In [6]:
P_A1_25.tail(5) # last N rows

,\n \n Núm Albarán,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Agua\n 2\n AGUA RECICLA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3,Aditivo\n 4\n AURAMIX 269
23439,1,28/05/2025,ARIDO A-20,1.0,0.0,0.0,1170.0,350.0,510.0,0.0,0.0,0.00,0.00,0.00
23440,Dosif. m3,NaN,NaN,NaN,0.0,0.0,1170.0,350.0,510.0,0.0,0.0,0.00,0.00,0.00
23441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23442,Total,NaN,NaN,55718.0,14293807.5,28072.0,66512253.0,22708604.0,23602593.5,7500793.0,0.0,9309.18,49182.26,89046.77
23443,Total m3,NaN,NaN,NaN,256.5,0.5,1193.7,407.6,423.6,134.6,0.0,0.20,0.90,1.60


In [7]:
P_A1_25.sample(6) # random N rows. It is runned multiple times to check whether the random rows match the source database records.

,\n \n Núm Albarán,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Agua\n 2\n AGUA RECICLA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3,Aditivo\n 4\n AURAMIX 269
2388,35235,03/12/2025,HA-25/B/20/XC2/1,10.0,2767.0,0.0,11573.0,2882.0,6223.0,1275.0,0.0,0.0,12.50,18.0
23338,Dosif. m3,NaN,NaN,NaN,222.0,0.0,1353.2,599.3,0.0,136.7,0.0,0.0,1.00,0.9
2992,Dosif. m3,NaN,NaN,NaN,276.7,0.0,1211.1,283.3,629.1,127.8,0.0,0.0,1.20,1.8
11139,32318,30/07/2025,HA-25/B/20/IIa/1,7.0,1939.0,0.0,8059.0,1929.0,4193.0,1018.0,0.0,0.0,6.75,15.6
16005,30696,13/05/2025,HA-30/B/20/XC2/1,10.0,2899.0,0.0,10952.0,3235.0,5762.0,1449.0,0.0,0.0,11.55,18.7
19241,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# How many of the rows has NaN values in all columns?
print("Count of rows with NaN values in all columns:", P_A1_25.isna().all(axis=1).sum())

Count of rows with NaN values in all columns: 7814


---

### Cleaning

In [9]:
# Deleting all rows where the values in all columns are NaN.
P_A1_25.dropna(how="all", inplace=True)

In [10]:
P_A1_25.shape

(15630, 14)

In [11]:
# How many of the rows has "Dosif.m3" in the first column?
count = (P_A1_25.iloc[:, 0] == "Dosif. m3").sum()
print(f"Count of rows with 'Dosif.m3' in the first column: {count}")

Count of rows with 'Dosif.m3' in the first column: 7814


In [12]:
# Deleting all rows where the first column contains "Dosif.m3" and "Dosif.m3.1"
P_A1_25.drop(P_A1_25[P_A1_25.iloc[:, 0] == "Dosif. m3"].index, inplace=True)

In [13]:
P_A1_25.shape
# 15630 - 7814 = 7816 rows remaining

(7816, 14)

In [14]:
# Droping last 2 rows because contain the total sum of the column values total consumption and total m3. We will use them as a references when we will sum in the final dataframe.
P_A1_25.drop(P_A1_25.tail(2).index, inplace=True)
# Dropping the first column "delivery note number" because does not much with the invoice system. The number of those are autogenerated but there is not relation with another business indicators.
P_A1_25.drop(P_A1_25.columns[0], axis=1, inplace=True)
# Dropping the column "Agua 2 AGUA" because all values are 0.00.
P_A1_25.drop("Agua\n 2\n AGUA RECICLA", axis=1, inplace=True)

In [15]:
# After dropping the rows identified as not useful any longer. 
# Rebuild a consecutive index: 0, 1, 2, 3, ...
P_A1_25.reset_index(drop=True, inplace=True)

In [16]:
print(P_A1_25.index[:10]) # slices the first 10 elements from that index
print(P_A1_25.shape)

RangeIndex(start=0, stop=10, step=1)
(7814, 12)


In [17]:
P_A1_25.sample

<bound method NDFrame.sample of      \n \n Fecha           \n \n Fórmula  \n \n Volumen  \
0     22/01/2026          SECOMD-200/S/4            6.0   
1     22/01/2026          HM-20/B/20/I/1            6.0   
2     21/01/2026          HM-20/B/20/I/1            4.0   
3     21/01/2026         HM-20/B/20/X0/1           10.0   
4     21/01/2026        HA-25/B/20/IIa/1            4.0   
...          ...                     ...            ...   
7809  17/06/2025        HA-25/B/20/XC1/1           10.0   
7810  29/05/2025  IMPRESO.HA-25/B/20/IIa            7.0   
7811  29/05/2025  IMPRESO.HA-25/B/20/IIa            7.0   
7812  28/05/2025        HA-25/B/14/IIa/1            1.0   
7813  28/05/2025              ARIDO A-20            1.0   

      Cemento\n 1\n CEM I 42,5R  Cemento\n 7\n CEM IV/A(V)  \
0                        1198.0                        0.0   
1                        1213.0                        0.0   
2                         807.0                        0.0   
3          

In [18]:
print(P_A1_25.columns.tolist()) # list of all column names

['\n \n Fecha', '\n \n Fórmula', '\n \n Volumen', 'Cemento\n 1\n CEM I 42,5R', 'Cemento\n 7\n CEM IV/A(V)', 'Arido\n 1\n ARENA 0-4', 'Arido\n 2\n ARIDO 8-14', 'Arido\n 3\n ARIDO 12-20', 'Agua\n 1\n AGUA', 'Aditivo\n 1\n COMPLAST MR-', 'Aditivo\n 3\n DYNAMON SX-3', 'Aditivo\n 4\n AURAMIX 269']


In [19]:
# Deleting columns in 11_P_A1_cleaning (deleted notebook). Why? because this code it can not apply to the function since that every plant has different columns names. This notebook is for exploring and set the steps for the first cleaning function to apply all of df.
# P_A1_25.rename(columns={"\n \n Fecha": "date", "\n \n Fórmula": "product", "\n \n Volumen": "volume", "Cemento\n 1\n CEM I 42,5R": "cement_42_5R", "Cemento\n 7\n CEM IV/A(V)": "cement_IV_AV", "Arido\n 1\n ARENA 0-4": "arid_04", "Arido\n 2\n ARIDO 8-14": "arid_8_14", "Arido\n 3\n ARIDO 12-20": "arid_12_20", "Agua\n 1\n AGUA": "water", "Aditivo\n 1\n COMPLAST MR-": "complast_mr", "Aditivo\n 3\n DYNAMON SX-3": "dynamon_sx3", "Aditivo\n 4\n AURAMIX 269": "auramix_269"}, inplace=True)

In [20]:
P_A1_25.info()                 

<class 'pandas.DataFrame'>
RangeIndex: 7814 entries, 0 to 7813
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   
 
 Fecha                 7814 non-null   str    
 1   
 
 Fórmula               7814 non-null   str    
 2   
 
 Volumen               7814 non-null   float64
 3   Cemento
 1
 CEM I 42,5R   7814 non-null   float64
 4   Cemento
 7
 CEM IV/A(V)   7814 non-null   float64
 5   Arido
 1
 ARENA 0-4       7814 non-null   float64
 6   Arido
 2
 ARIDO 8-14      7814 non-null   float64
 7   Arido
 3
 ARIDO 12-20     7814 non-null   float64
 8   Agua
 1
 AGUA             7814 non-null   float64
 9   Aditivo
 1
 COMPLAST MR-  7814 non-null   float64
 10  Aditivo
 3
 DYNAMON SX-3  7814 non-null   float64
 11  Aditivo
 4
 AURAMIX 269   7814 non-null   float64
dtypes: float64(10), str(2)
memory usage: 935.3 KB


In [21]:
print(P_A1_25["\n \n Fecha"].head(15).tolist())

['22/01/2026', '22/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026', '21/01/2026']


In [22]:
print(P_A1_25["\n \n Fecha"].dtype)

str


In [23]:
# Converting the "date" column to datetime format.
P_A1_25["\n \n Fecha"] = pd.to_datetime(P_A1_25["\n \n Fecha"], errors="coerce", dayfirst=True)

In [24]:
print(P_A1_25["\n \n Fecha"].isna().sum())
print(P_A1_25["\n \n Fecha"].head(5))

0
0   2026-01-22
1   2026-01-22
2   2026-01-21
3   2026-01-21
4   2026-01-21
Name: \n \n Fecha, dtype: datetime64[us]


In [25]:
# Formatting the "date" column to "YYYY-MM-DD" format.
P_A1_25["\n \n Fecha"] = P_A1_25["\n \n Fecha"].dt.strftime("%Y-%m-%d")

In [26]:
P_A1_25.head()

,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3,Aditivo\n 4\n AURAMIX 269
0,2026-01-22,SECOMD-200/S/4,6.0,1198.0,0.0,11315.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-01-22,HM-20/B/20/I/1,6.0,1213.0,0.0,7337.0,2116.0,3247.0,552.0,0.0,0.0,12.0
2,2026-01-21,HM-20/B/20/I/1,4.0,807.0,0.0,4949.0,1360.0,2140.0,415.0,0.0,0.0,8.0
3,2026-01-21,HM-20/B/20/X0/1,10.0,2021.0,0.0,11603.0,3522.0,5802.0,1179.0,0.0,0.0,13.1
4,2026-01-21,HA-25/B/20/IIa/1,4.0,1105.0,0.0,4696.0,1176.0,2508.0,507.0,0.0,5.1,7.2


In [27]:
print(P_A1_25["\n \n Fecha"].dtype)
print(P_A1_25["\n \n Fecha"].head())

str
0    2026-01-22
1    2026-01-22
2    2026-01-21
3    2026-01-21
4    2026-01-21
Name: \n \n Fecha, dtype: str


In [28]:
# Deleting rows where the year is 2026.
P_A1_25 = P_A1_25[~P_A1_25["\n \n Fecha"].str.startswith("2026")]

In [29]:
# Reestablishing index.
P_A1_25.reset_index(drop=True, inplace=True)

In [30]:
P_A1_25.sample(5)

,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3,Aditivo\n 4\n AURAMIX 269
4790,2025-05-26,HA-30/F/14/XC4/1,10.0,3201.0,0.0,11842.0,8015.0,0.0,1372.0,0.0,12.85,25.6
6078,2025-03-21,HL-150/B/20/1,10.0,1498.0,0.0,12283.0,3481.0,4963.0,1045.0,0.0,0.00,9.7
4462,2025-06-05,HA-25/B/20/IIa/1,7.0,1928.0,0.0,7962.0,1909.0,4163.0,1058.0,0.0,6.85,15.6
914,2025-11-13,HA-30/B/20/XC2/1,5.5,1587.0,0.0,6359.0,1773.0,3135.0,796.0,0.0,5.60,12.8
7127,2025-01-21,IMPRESO.HA-25/B/14/IIa,2.5,695.0,0.0,2958.0,2068.0,0.0,367.0,0.0,4.85,0.0


Not adding rename columns to the function because each df has different columns.

**Handoff to 01_testing**
        This notebook builds the cleaning steps for the P_A1_25 dataset.
        The next notebook applies the same function again and checks that the cleaned output matches the manual build.

---